# **PromptKaban: Semantic Search Engine**

This notebook builds and evaluates a **semantic search engine** over a corpus of **20,000 AI prompts**. 

Users can ask questions in plain language, and the engine finds the best matching prompts using:

- Embeddings from **nomic-embed-text-v1.5**
- A spectral graph index from **ArrowSpace**
- An engagement-aware reranker (MMR + salience from upvotes, likes, reputation, views)

We start by looking at the raw data to understand what the model sees.

## **1. Data preview**

The dataset is stored as JSON:

- `dataset.json`: 20,000 prompts with content and metadata  
- `queries.json`: evaluation queries, each with an `expected_prompt_id`

Below we sample one random prompt and one random query to check the structure.

In [ ]:
!jq ".[$(jq 'keys[]' ../data/dataset.json | shuf -n 1)]" ../data/dataset.json

In [ ]:
!jq ".[$(jq 'keys[]' ../data/queries.json | shuf -n 1)]" ../data/queries.json

The sample prompt `pk_16762` shows the usual fields we will use: 
- Text fields for meaning: `title`, `content`, `category`, `subcategory`, `tags` 
- Signals for quality: `upvotes`, `likes`, `author_reputation`, `views` 

The sample query (like `q_02`) shows how we check things. For each `query_text`, we know which prompt ID (`expected_prompt_id`) to find.

## 2. Structured document representation

If we only use the raw `content` field, many prompts are too short or unclear to match real queries well. 

To fix this, we build a single **`doc_string`** for each prompt that includes all useful fields:

- Title
- Category and subcategory
- Difficulty
- Tags
- Full content

In [ ]:
import json 
from pathlib import Path 
from tinydb import TinyDB, Query
import re
import numpy as np
import arrowspace_tuner


db = TinyDB('db.json')
ROOT = Path.cwd().parent 
dataset_path = ROOT / "data" / "dataset.json"
query_path = ROOT / "data" / "nomic_embs" / "queries_emb_768.npy"
query_index_path = ROOT / "data" / "nomic_embs" / "queries_index.json"

DOC_PREFIX = "search_document: "

In [ ]:
with open(dataset_path, "r") as f:
    dataset = json.load(f)

In [ ]:
prompts = Query()

In [ ]:
def clean(text: str) -> str:
    """Remove {{placeholder}} tokens, collapse whitespace."""
    return re.sub(r"\s+", " ", re.sub(r"\{\{[^}]+\}\}", "", text)).strip()


def is_weak_title(el: dict) -> bool:
    """True when title has ≤4 words or shares no token with its own tags/category."""
    t = el["title"].lower()
    signals = el.get("tags", []) + [el.get("category", ""), el.get("subcategory", "")]
    has_signal = any(s.lower().replace("-", " ") in t for s in signals if s)
    return len(el["title"].split()) <= 4 or not has_signal


def extract_structured(el: dict) -> str:
    """
    Richest form: explicit field labels + full content.
    Named fields (Title:, Category:, Tags:) guide nomic attention to distribute
    semantic load evenly across ALL Matryoshka bands.
    Primary fix for Type-A failures (branding, coaching, communication, audit):
    engine is completely blind — needs every contextual signal available.
    """
    title  = el["title"].strip()
    tags   = ", ".join(el["tags"])
    body   = clean(el["content"])
    cat    = el["category"]
    subcat = el.get("subcategory", "")
    diff   = el.get("difficulty", "general")
    ph     = el.get("placeholders", [])
    ph_str = f"Variables: {', '.join(ph)}.\n" if ph else ""

    return (
        DOC_PREFIX
        + f"Title: {title}\n"
          f"Category: {cat} > {subcat}\n"
          f"Difficulty: {diff}\n"
          f"Tags: {tags}\n"
        + ph_str
        + body
    ).strip()


In [ ]:
for el in dataset:
    prompt = extract_structured(el)
    db.insert({'id' : el["id"], 'doc_string': prompt})

In [ ]:
db.search(prompts.id == "pk_03138")

This format gives the embedding model a lot more signal than the raw content alone. Every prompt is stored in TinyDB as `{id, doc_string}`, ready for the embedding step.

## **3. Load embeddings and build ArrowSpace**

We now move from **raw text** to **vector space** and build the spectral index.

### **3.1 Data pipeline**

The main work (encoding and saving embeddings) is done once:

**Offline steps:**

1. We store all prompts in TinyDB (`db.json`) as `{"id": ..., "doc_string": ...}` for 20k records.

2. `scripts/nomic_corpus_embs.py` reads those records, runs `nomic-embed-text-v1.5` on each `doc_string`, and saves:
    - `embeddings_nomic_structured_768d_raw.npy`  → `(N, 768)` float32
    - `embeddings_nomic_structured_768d_ids.npy`  → `(N,)` str, same row order
    - `embeddings_registry.json` → tells us, for each ID, which row and which file to load

3. In this notebook we load those `.npy` files and call `ArrowSpaceBuilder.build(...)` to create:
    - `aspace` → the ArrowSpace index (with one λ value per prompt)
    - `gl` → the Graph Laplacian reused at query time

We always keep IDs and embeddings in the **same row order**, so `ids[i]` matches `embs[i]` and `aspace.lambdas()[i]`. 
That alignment is checked with an assert.


**Memory note (float64 in RAM)**
| Prompts | 256d  | 512d  | 768d   |
|---------|-------|-------|--------|
| 20k     | 40 MB | 80 MB | 120 MB |
| 100k    | 200 MB| 400 MB| 600 MB |
| 500k    | 1 GB  | 2 GB  | 3 GB   |
| 1M      | 2 GB  | 4 GB  | 6 GB   |

ArrowSpace keeps an internal copy of the matrix, so you can roughly **double** these numbers. 

For 1M prompts, 256d fits on a 16 GB machine; 768d is more comfortable on 32 GB. The sparse Laplacian adds only ~50 MB on top.

In [ ]:
import numpy as np
from pathlib import Path

EMBS_DIR = ROOT / "data" / "nomic_embs" 
DIM      = 768  

emb_path = EMBS_DIR / f"embeddings_nomic_structured_{DIM}d_raw.npy"
ids_path = EMBS_DIR / f"embeddings_nomic_structured_{DIM}d_ids.npy"

# ArrowSpace requires float64
embs = np.load(emb_path).astype(np.float64)
ids  = list(np.load(ids_path))

# alignment guard — must never fail
assert embs.shape[0] == len(ids), \
    f"Shape mismatch: embs {embs.shape[0]} rows vs {len(ids)} ids"

print(f"Embeddings : {embs.shape}  dtype={embs.dtype}")
print(f"IDs        : {len(ids)} entries")
print(f"RAM in use : {embs.nbytes / 1e9:.2f} GB")
print(f"Sample     : ids[0]={ids[0]!r}  embs[0,:4]={embs[0, :4]}")

The corpus embeddings load as a `(20_000, 768)` matrix in `float64`, using about **0.12 GB**.  
The check `len(ids) == embs.shape[0]` makes sure that every row in the matrix has a matching prompt ID. This is necessary before we build any index.

### **3.2 ArrowSpace graph parameters tuning**

ArrowSpace builds a k‑NN graph and a graph Laplacian over the feature space. It gives each prompt a **Rayleigh quotient λ** that shows how central or peripheral it is in the semantic manifold.

To avoid hand-tuning graph parameters, we use a query free optimization function with a Bayesian optimization to search over:

- `eps` ∈ [0.5, 2.0]: neighbourhood radius  
- `k`: effective number of nearest neighbours  


It runs 15 tests on a 5,000‑sample and stores results in a local SQLite DB for reproducibility (`arrowspace_tuning.db`).


In [ ]:
from arrowspace_tuner import EpsTuner

tuner = EpsTuner(
    n_trials  = 15,
    sample_n  = 5_000,
    eps_low   = 0.5,
    eps_high  = 2.0,
    tau_low   = 0.42,
    tau_high  = 1.0,
    n_probe   = 50,
    storage   = "sqlite:///arrowspace_tuning.db",
)

In [ ]:
aspace, gl = tuner.fit(embs)
tuner.save_report()

In [ ]:
graph_params = tuner.load_best_params()

graph_params

`aspace` now holds the ArrowSpace index, and `gl` holds the corresponding graph Laplacian.  

Every prompt has:
- an embedding vector in `embs[i]`
- a prompt ID in `ids[i]`
- a spectral energy score λᵢ available via `aspace.lambdas()[i]`

In the next section we inspect the graph structure to understand how the dataset is wired in the spectral domain before running any retrieval experiments.

## **4. Dataset monitoring (spectral fingerprint)**

Before running any queries, we inspect how ArrowSpace has structured the corpus in the graph. 

This gives us a picture of where retrieval will be easy and where it will be hard.

Two things to look at:
- **λ values** (Rayleigh quotient): one number per prompt, ranging from 0 to 1.  
    **Low λ** = the prompt sits in a dense, well-connected region.  
    **High λ** = the prompt is isolated and harder to find.

- **Node degree**: how many neighbours each prompt centroid has in the graph of the feature space.  
    **High degree** = semantic hub. 
    **Low degree** = niche topic.

In [ ]:
lambdas = aspace.lambdas()
print("aspace_base (semantic only)")
print(f"  λ  min={min(lambdas):.6f}  max={max(lambdas):.6f}  "
      f"mean={sum(lambdas)/len(lambdas):.6f}")

The **λ distribution** is **right-skewed**: most prompts have low λ (they live in dense clusters), while a long tail reaches λ = 1.0. The mean of **0.124** confirms that the majority of the feature space is well-connected, but that tail is exactly where a plain cosine search would struggle.

### **4a. Graph Laplacian manifold**

The plot below turns the graph structure into a 3D surface.

- Each point on the flat plane represents a region in the embedding space (laid out using PCA on the 768d vectors).
- **Height (Z)** = node degree at that location. How many connections each  centroid has in the k-NN graph.
- **Red dots** = the top 15% highest-degree nodes, the semantic hubs of the corpus.
- **Blue hollows** = sparse, low-degree regions, niche topics with few neighbours.

The surface is built by interpolating degrees across the PCA plane and smoothing with a Gaussian filter, so what you see is a continuous "landscape" of connectivity across the entire 20,000-prompt corpus.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from sklearn.decomposition import PCA
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter

rows = embs 

def gl_to_scipy(gl) -> sp.csr_matrix:
    raw = gl.to_csr()
    return sp.csr_matrix(
        (np.asarray(raw[0]), np.asarray(raw[1]), np.asarray(raw[2])),
        shape=gl.shape()
    )

L       = gl_to_scipy(gl)
n_nodes = L.shape[0]
degrees = np.array(L.diagonal(), dtype=np.float64)

p95   = np.percentile(degrees, 95)
p05   = np.percentile(degrees, 5)
z_pts = np.clip(degrees, p05, p95)

try:
    if 'rows' not in dir() or rows is None:
        raise NameError("rows not defined")
    embs_2d      = np.array(rows[:n_nodes], dtype=np.float32)
    pca          = PCA(n_components=2)
    xy           = pca.fit_transform(embs_2d)
    x_pts, y_pts = xy[:, 0], xy[:, 1]
    var          = pca.explained_variance_ratio_
    x_label      = f'PC1 ({var[0]*100:.1f}%)'
    y_label      = f'PC2 ({var[1]*100:.1f}%)'
    subtitle     = f'PC1 {var[0]*100:.1f}%  PC2 {var[1]*100:.1f}%'

except NameError:
    k_eig        = min(4, n_nodes - 1)
    vals, vecs   = eigsh(L.astype(np.float64), k=k_eig, which='SM', tol=1e-5, maxiter=3000)
    order        = np.argsort(vals)
    vecs         = vecs[:, order]
    x_pts        = vecs[:, 1]
    y_pts        = vecs[:, 2]
    var          = None
    x_label      = f'Fiedler v (λ₂={vals[order[1]]:.3f})'
    y_label      = f'Spectral v (λ₃={vals[order[2]]:.3f})'
    subtitle     = 'spectral embedding — Fiedler (λ₂) × λ₃'

grid_res = 120
xi = np.linspace(x_pts.min(), x_pts.max(), grid_res)
yi = np.linspace(y_pts.min(), y_pts.max(), grid_res)
Xi, Yi = np.meshgrid(xi, yi)

Zi      = griddata((x_pts, y_pts), z_pts, (Xi, Yi), method='cubic')
Zi_near = griddata((x_pts, y_pts), z_pts, (Xi, Yi), method='nearest')
Zi[np.isnan(Zi)] = Zi_near[np.isnan(Zi)]
Zi = gaussian_filter(Zi, sigma=2.5)

colorscale = [
    [0.0,  '#1a3a6b'],
    [0.25, '#4a90d9'],
    [0.5,  '#f5f5f5'],
    [0.75, '#e05c3a'],
    [1.0,  '#8b0000'],
]

surface = go.Surface(
    x=Xi, y=Yi, z=Zi,
    colorscale=colorscale,
    opacity=1.0,
    showscale=True,
    colorbar=dict(
        title=dict(text='Curvature (Lᵢᵢ)', font=dict(size=12)),
        thickness=16, x=1.01, tickfont=dict(size=10),
    ),
    contours=dict(
        z=dict(
            show=True,
            usecolormap=True,
            highlightcolor='rgba(255,255,255,0.6)',
            project_z=True,
            start=float(Zi.min()),
            end=float(Zi.max()),
            size=(Zi.max() - Zi.min()) / 12,
        )
    ),
    lighting=dict(ambient=0.7, diffuse=0.9, specular=0.2,
                  roughness=0.6, fresnel=0.1),
    lightposition=dict(x=1000, y=1000, z=2000),
    hoverinfo='skip',
    name='Manifold',
)

top_idx = np.where(degrees > np.percentile(degrees, 85))[0]
node_scatter = go.Scatter3d(
    x=x_pts[top_idx], y=y_pts[top_idx], z=z_pts[top_idx] + 1,
    mode='markers',
    marker=dict(
        size=5,
        color=degrees[top_idx],
        colorscale=colorscale,
        cmin=p05, cmax=p95,
        opacity=1.0,
        line=dict(width=0.8, color='white'),
    ),
    text=[f'Node {i}<br>degree={degrees[i]:.2f}' for i in top_idx],
    hoverinfo='text',
    name='High-degree hubs (top 15%)', 
)

fig = go.Figure(
    data=[surface, node_scatter],
    layout=go.Layout(
        title=dict(
            text=(
                f'<b>Graph Laplacian Manifold ({n_nodes} nodes)</b><br>'
                f'<i style="font-size:11px; color:#a0a8c0;">'
                f'Node connectivity as a proxy for local manifold curvature ({subtitle})'
                f'</i>'
            ),
            x=0.5,
            font=dict(size=20, color='#e2e6f0'),
        ),
        scene=dict(
            xaxis=dict(title=x_label,
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            yaxis=dict(title=y_label,
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            zaxis=dict(title='Node Degree (Lᵢᵢ)',  
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            bgcolor='rgb(10,10,20)',
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.9),
                        up=dict(x=0, y=0, z=1)),
            aspectmode='manual',
            aspectratio=dict(x=1.6, y=1.6, z=0.55),
        ),
        paper_bgcolor='rgb(12,12,22)',
        font=dict(color='#e2e6f0', family='system-ui, sans-serif'),
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(0,0,0,0.5)',
                    font=dict(size=11)),
        margin=dict(l=0, r=0, t=110, b=40),  # ← updated
        height=700,
    )
)

fig.show(renderer="vscode")

## **Graph Laplacian Manifold Interpretation**

This 3D surface visualises the **spectral geometry** of the feature space induced by the Graph Laplacian **L** built over the dataset.

**Lᵢᵢ = degᵢ** serves as a discrete curvature estimator on the embedding features manifold.

**Axes:** 
- **X / Y**: first two principal components of the node embeddings (PCA projection, ~8.4% total variance explained). 
Low explained variance is expected in high-dimensional semantic spaces; PCA is used here for visualisation only, all spectral computations operate in the full high-dimensional space. 

- **Z**: diagonal entry **Lᵢᵢ** of the Laplacian, equal to the degree of node *i*. High values indicate nodes with many strong semantic neighbours, regions of high local connectivity and, by analogy with the Laplace–Beltrami operator, **high discrete curvature**.

From the graph, we can deduct the following: 
- **Peaks (red)** highlights dense **semantic hubs**, concepts or documents that are highly connected in the similarity graph. These are the most "central" regions of the manifold.
- **Valleys (blue)** highlights **sparse**, peripheral nodes with few neighbours, isolated or niche content.
- **Scattered red dots** highlights **top 15% highest-degree nodes** (hub nodes), explicitly surfaced above the manifold for inspection.

Regions of high curvature contain a dense concentration of meaning and are where retrieval, clustering, and ranking decisions are most affected by small changes in the embedding space. In the context of retrieval, hub nodes (with high **Lᵢᵢ**) serve as semantic anchors. **Boosting** their importance during the ranking process has been shown to significantly enhance recall for low-frequency queries.

### **4b. λ Distribution (Spectral Fingerprint)**

The two panels below show the **Rayleigh quotient** distribution for **20,000 prompts**. 
- **Top panel**: A histogram that cuts off at p60 to show the main shape. If it didn’t cut off, the spike at λ ≈ 0 would flatten everything else. 

- **Bottom panel**: An ECDF that shows the full range from 0 to 1, with p25, median, and p75 marked.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Using your stats from the image_f3e3db.png terminal output
# mean: 0.128270, median: 0.045436, p95: ~0.45

# --- 1. Clipping Strategy ---
x_limit = np.percentile(lambdas, 60)  # Focus on the 95% bulk
lambdas_clipped = lambdas[lambdas <= x_limit]
tail_count = len(lambdas) - len(lambdas_clipped)
tail_pct = (tail_count / len(lambdas)) * 100

# Higher resolution for the bulk of the data
bins = np.linspace(0, x_limit, 200)
h, edges = np.histogram(lambdas, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2

# --- 2. Figure Construction ---
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        f"λ Density — Bulk Distribution (Clipped at p60)",
        "ECDF — Linear Scale with Progress Zones"
    ),
    vertical_spacing=0.15,
)

# Row 1: Histogram (Linear but Focused)
fig.add_trace(go.Bar(
    x=centers, y=h,
    name='Base Distribution',
    marker=dict(
        color=centers,
        colorscale='Viridis',
        showscale=False,
        line_width=0
    ),
    hovertemplate='λ: %{x:.4f}<br>Count: %{y}<extra></extra>',
), row=1, col=1)

# Explicit "Outlier" Indicator
fig.add_annotation(
    x=x_limit, y=max(h)*0.8,
    text=f"▶ {tail_count:,} items ({tail_pct:.1f}%)<br>continue in tail to max=1.0",
    showarrow=True, arrowhead=2, ax=-60, ay=0,
    bgcolor="rgba(192,57,43,0.8)", bordercolor="#c0392b",
    font=dict(size=11, color="white"), row=1, col=1
)

# Row 2: ECDF
sl = np.sort(lambdas)
ey = np.arange(1, len(sl)+1) / len(sl)

fig.add_trace(go.Scatter(
    x=sl, y=ey, mode='lines',
    line=dict(color='#4a90d9', width=3),
    fill='tozeroy', fillcolor='rgba(74,144,217,0.05)',
    name='Cumulative',
    hovertemplate='λ ≤ %{x:.4f}<br>Percentile: %{y:.1%}<extra></extra>'
), row=2, col=1)

# Vertical threshold lines for p25, Median, p75
for val, label, col in [
    (np.percentile(lambdas, 25), 'p25', '#4ecdc4'),
    (np.median(lambdas), 'Median', '#ffcc00'),
    (np.percentile(lambdas, 75), 'p75', '#ff6b6b'),
]:
    fig.add_vline(x=val, line_dash='dot', line_color=col, line_width=1, row=2, col=1)
    fig.add_annotation(x=val, y=0.1, text=label, font=dict(color=col, size=10), 
                       showarrow=False, textangle=-90, xshift=10, row=2, col=1)

# --- 3. Layout & Style ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='rgb(10, 10, 15)',
    plot_bgcolor='rgb(15, 15, 25)',
    height=800,
    title=dict(
        text=f"Spectral Fingerprint: {len(lambdas):,} Samples",
        font=dict(size=18, color="#0E87FF"), x=0.5
    ),
    margin=dict(t=120)
)

# Set the focus range
fig.update_xaxes(range=[0, x_limit], row=1, col=1, title_text="λ (Rayleigh quotient)")
fig.update_xaxes(range=[0, 1.0], row=2, col=1, title_text="λ (Full Range for ECDF)")
fig.update_yaxes(title_text="Frequency", row=1, col=1, gridcolor='rgba(255,255,255,0.05)')
fig.update_yaxes(title_text="Percentile", tickformat='.0%', row=2, col=1, gridcolor='rgba(255,255,255,0.05)')

fig.show()

### **λ Distribution Interpretation**
The distribution exhibits a strong right-skewed asymmetry, with many data points close to λ ≈ 0. 

The top graph **(λ Density)** shows the main density at p60: there is a big spike at λ ≈ 0 with about 5,000 counts, while the rest is mostly flat. This shows that most prompts have a Rayleigh quotient close to zero. ALso, there are 8,000 items (40%) above the p60 level that go into a long tail reaching λ = 1.0. 

The bottom part **(ECDF)** shows the shape from **0 to 1**. The curve starts at about 42% at λ = 0, highlighting that almost half of the data has Rayleigh quotients close to zero. Both the p25 and the median are within the first 0.05, while the p75 only reaches λ ≈ 0.21. The curve then flattens slowly, showing a small but steady tail.

Two types of retrieval emerge from this shape: 
- **Bulk (λ < p60)**: connected prompts where cosine nearest neighbor search works well and quickly.
- **Tail (λ > p60)**: less connected prompts where the Rayleigh quotient does not correspond to the cosine rank order, and finding them requires λ-sensitive re-ranking. 

The tail has a sufficient number of **prompts (8,000)** to cause issues for specific queries, but is still dense enough that it does not cause the spectral score to influence the main cluster. This distribution leads to the engagement-sensitive re-ranker discussed in Section 5.

## **5. Search and Document Retrieval**

Once we have developed the spectral index and profiled the corpus, we start the retrieval workflow and compare **two classification algorithms**. 

We load the precomputed representation of each evaluation query, use **ArrowSpace** to perform a basic search, reclassify the same candidates using **MMR+Salience**, and display both lists side by side, with the reference prompt highlighted.

### **5.1 Data preview**

In [ ]:
dataset[0]

Each prompt has **two types of fields**:

**Semantic**: used to build the `doc_string` for embedding:
`title`, `content`, `category`, `subcategory`, `tags`, `difficulty`

**Engagement**:  used to compute the salience score:
`upvotes`, `likes`, `author_reputation`, `views`

### **5.2 Load evaluation queries**

In [ ]:

q_embs = np.load(query_path).astype(np.float64)
idx    = json.load(open(query_index_path))

row    = idx["q_09"]["row_index"]          # 8
query  = q_embs[row].astype(np.float64)

In [ ]:
ids_path = ROOT / "data" / "nomic_embs" / "embeddings_nomic_structured_768d_ids.npy"

### **5.3 Build dataset lookup and salience map**

In [ ]:
dataset_map = {r["id"]: r for r in dataset}

print(f"dataset_map built: {len(dataset_map)} records")
print(f"  sample keys: {list(dataset_map.keys())[:5]}")

We create a `dataset_map` to quickly access full prompt details using IDs. TinyDB only keeps the `id` and `doc_string`. 
The **salience score** combines four engagement signals: upvotes (0.35), likes (0.35), author reputation (0.20), and log-views (0.10). 

The final result is `salience_map`, which is a dictionary with prompt IDs as keys and values between 0 and 1.

In [ ]:
W_UP   = 0.35
W_LK   = 0.35
W_REP  = 0.20
W_VIEW = 0.10

def _norm(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)

# ── Canonical ids order must match embs row indices ──────────────────────────
# Load from numpy file so this cell and the visualiser share the same ordering.
ids     = list(np.load(ids_path))
records = [dataset_map[i] for i in ids if i in dataset_map]

# Diagnostic: warn if metadata is sparse
missing_ups  = sum(1 for r in records if r.get("upvotes")           is None)
missing_lks  = sum(1 for r in records if r.get("likes")             is None)
missing_rep  = sum(1 for r in records if r.get("author_reputation") is None)
missing_view = sum(1 for r in records if r.get("views")             is None)
print(f"Salience: {len(records)} records loaded")
print(f"  missing upvotes={missing_ups}  likes={missing_lks}  "
      f"reputation={missing_rep}  views={missing_view}")

upvotes    = _norm(np.array([r.get("upvotes", 0)           for r in records], dtype=float))
likes      = _norm(np.array([r.get("likes", 0)             for r in records], dtype=float))
reputation = _norm(np.array([r.get("author_reputation", 0) for r in records], dtype=float))
views      = _norm(np.log1p(np.array([r.get("views", 0)    for r in records], dtype=float)))

# FIX: weighted sum of [0,1] arrays is already in [0,1] — no second _norm needed.
# The original double normalisation forced the top item to 1.0 regardless of spread.
salience_arr = W_UP * upvotes + W_LK * likes + W_REP * reputation + W_VIEW * views

salience_map = {ids[i]: float(salience_arr[i]) for i in range(len(ids))}
print(f"  salience range: [{salience_arr.min():.4f}, {salience_arr.max():.4f}]  "
      f"mean={salience_arr.mean():.4f}")

### 5.4 Spot-check: Base vs MMR+Salience

We run all tests through both systems and show the results next to each other. 
For each test, `aspace.search()` gives the basic ranking, and `mmr_rerank()` changes the order using a special score. 
We display the top 3 and bottom 3 from the top 10, with the main prompt marked in green.

In [ ]:
# ── MMR: Maximal Marginal Relevance + Salience ────────────────────────────────
# FIX: LAM=0.7 (was 1.0, which silently disabled diversity — lam=1.0 makes
# (1-lam)*sim_to_selected = 0, degenerating to a plain salience re-sort).
LAM        = 0.7   # 1.0=no diversity  0.7=balanced  0.0=max diversity
SAL_WEIGHT = 0.30
N_RETRIEVE = 10


def mmr_rerank(candidates, embs, ids, lam=LAM, k=N_RETRIEVE,
               sal_weight=SAL_WEIGHT, sal_map=None):
    """
    MMR reranker with salience blend.

    candidates : list of (row_index, cosine_score)  — raw aspace output
    embs       : np.ndarray  shape (N, D), indexed by row_index
    ids        : list of prompt_ids, indexed by row_index
    sal_map    : dict  prompt_id → salience in [0, 1]

    lam regimes:
      1.0  → pure salience re-sort (diversity disabled)
      0.7  → classic MMR (default)
      0.0  → maximum diversity

    Complexity: O(k * |candidates| * D) — fine for k~10, D~384.
    """
    # FIX: guard against ids/embs ordering mismatch (silent wrong results otherwise)
    assert len(ids) == len(embs), (
        f"ids/embs length mismatch: {len(ids)} ids vs {len(embs)} emb rows. "
        "Ensure both come from the same numpy-loaded ids_path."
    )
    if sal_map is None:
        sal_map = {}

    def relevance(i):
        ri  = candidates[i][0]   # row index into embs / ids
        cos = candidates[i][1]   # cosine score
        pid = ids[ri]            # prompt id
        sal = sal_map.get(pid, 0.0)
        return (1.0 - sal_weight) * cos + sal_weight * sal

    selected, remaining = [], list(range(len(candidates)))

    while len(selected) < k and remaining:
        if not selected:
            best = max(remaining, key=relevance)
        else:
            sel_embs = np.array([embs[candidates[i][0]] for i in selected])

            def mmr_score(i):
                e     = embs[candidates[i][0]]
                norms = np.linalg.norm(sel_embs, axis=1) * np.linalg.norm(e) + 1e-9
                sim_to_selected = np.max(sel_embs @ e / norms)
                return lam * relevance(i) - (1 - lam) * sim_to_selected

            best = max(remaining, key=mmr_score)

        selected.append(best)
        remaining.remove(best)

    return [candidates[i] for i in selected]

In [ ]:
from IPython.display import display, HTML
import numpy as np

# ids already loaded in Cell 1 (canonical numpy order)
id_to_row  = {pk: i for i, pk in enumerate(ids)}

ALPHA        = 0.6
N_RETRIEVE   = 10
SPOT_QUERIES = list(idx.keys())
pk_meta      = {item["id"]: item for item in dataset}

# ── palette ───────────────────────────────────────────────────────────────────
BG   = "#0f1117"; SURF = "#1a1d27"; SURF2 = "#22263a"
BDR  = "#2e3350"; TXT  = "#e2e6f0"; MUT   = "#7a82a0"
ASP  = "#a86fdf"; GRN  = "#6daa45"; RED   = "#dd6974"; ORG  = "#fdab43"
TEAL = "#4f98a3"; YLW  = "#e8b84b"; AMBER = "#fdab43"; BLUE = "#4a90d9"
TARGET_BORDER = "#6daa45"  # FIX: uniform green in both columns (was column colour)


def prompt_snippet(pk, max_chars=140):
    item = pk_meta.get(pk, {})
    body = item.get("content", "—")
    snip = body[:max_chars] + ("…" if len(body) > max_chars else "")
    return snip, item.get("title", pk), item.get("category", ""), item.get("difficulty", "")


def _rank_badge(rank, total):
    col = GRN if rank <= 2 else (RED if rank >= total - 1 else MUT)
    return (f'<span style="background:{SURF2};color:{col};font-family:monospace;'
            f'font-size:12px;font-weight:700;padding:2px 8px;border-radius:5px">#{rank}</span>')


def _target_badge(rank, total, not_found=False):
    if not_found:
        return (f'<span style="background:{SURF2};color:{RED};font-family:monospace;'
                f'font-size:11px;font-weight:700;padding:2px 8px;border-radius:5px">'
                f'✗ not in top-{total}</span>')
    col = GRN if rank <= 3 else (YLW if rank <= total // 2 else RED)
    return (f'<span style="background:{SURF2};color:{col};font-family:monospace;'
            f'font-size:11px;font-weight:700;padding:2px 8px;border-radius:5px">'
            f'🎯 #{rank}/{total}</span>')


def _result_row(rank, pk, score, expected_pk, total, sal_map=None):
    snip, title, cat, diff = prompt_snippet(pk)
    is_target = pk == expected_pk
    row_bg    = "#162816" if is_target else SURF
    # FIX: TARGET border is uniform green in both columns, not column-coloured
    border_l  = f"border-left:3px solid {TARGET_BORDER}" if is_target else ""
    tgt_badge = (f' <span style="color:{GRN};font-size:10px;font-weight:700;'
                 f'background:#1e3a1e;padding:1px 5px;border-radius:3px">TARGET</span>'
                 if is_target else "")
    item = pk_meta.get(pk, {})
    ups  = item.get("upvotes", 0)
    lks  = item.get("likes", 0)
    sal  = sal_map.get(pk, 0.0) if sal_map else 0.0
    eng  = (f'<span style="color:{MUT};font-size:10px">▲{ups} ♥{lks} '
            f'<span style="color:{TEAL}">✦{sal:.2f}</span></span>')
    return f"""
<tr style="background:{row_bg};border-bottom:1px solid {BDR};{border_l}">
  <td style="padding:6px 10px;vertical-align:top;white-space:nowrap">
    {_rank_badge(rank, total)}
  </td>
  <td style="padding:6px 10px;vertical-align:top;max-width:200px">
    <code style="color:{ASP};font-size:11px">{pk}</code>{tgt_badge}<br>
    <span style="color:{TXT};font-weight:600;font-size:12px">{title}</span><br>
    <span style="color:{MUT};font-size:10px">{cat}</span><br>{eng}
  </td>
  <td style="padding:6px 10px;vertical-align:top;font-family:monospace;
      color:{MUT};font-size:11px;max-width:300px;word-break:break-word">{snip}</td>
  <td style="padding:6px 10px;vertical-align:middle;text-align:right;
      font-family:monospace;color:{ASP};font-size:12px;font-weight:700;
      white-space:nowrap">{score:.5f}</td>
</tr>"""


def _separator_row(hidden):
    return (f'<tr style="background:{SURF2}"><td colspan="4" style="padding:4px 10px;'
            f'color:{MUT};font-size:10px;text-align:center">·· {hidden} hidden ··</td></tr>')


def _normalise_hits(hits):
    """Accept (ri, sc) from aspace OR (ri, sem, sal, final) from salience reranker."""
    out = []
    for h in hits:
        if len(h) == 4:
            out.append((h[0], h[3]))   # final blended score
        else:
            out.append((h[0], h[1]))   # raw cosine
    return out


def _build_table(hits, ids, expected_pk, n_retrieve, label, color, badge_override=None,
                 sal_map=None):
    """Returns (complete_html_table_string, badge, total, target_rank)."""
    hits  = _normalise_hits(hits)
    top_n = hits[:n_retrieve]
    total = len(top_n)

    target_rank = next(
        (r for r, (ri, _) in enumerate(top_n, 1) if ids[ri] == expected_pk), None
    )

    top3          = top_n[:3]
    bot_start_idx = max(3, total - 3)
    bottom3       = top_n[bot_start_idx:]
    hidden        = max(0, total - 6)

    rows_html = ""
    for rank, (ri, sc) in enumerate(top3, 1):
        rows_html += _result_row(rank, ids[ri], sc, expected_pk, total, sal_map)
    if hidden > 0:
        rows_html += _separator_row(hidden)
    for rank, (ri, sc) in enumerate(bottom3, bot_start_idx + 1):
        rows_html += _result_row(rank, ids[ri], sc, expected_pk, total, sal_map)

    badge = badge_override or _target_badge(target_rank, total,
                                             not_found=(target_rank is None))

    # FIX: _col_header now returns a self-contained table (open + header row)
    # and _build_table closes it — helper is no longer broken in isolation
    header = f"""
<div style="background:{SURF2};padding:8px 12px;border-bottom:2px solid {color}">
  <span style="color:{color};font-weight:700;font-size:13px">{label}</span>
  <span style="float:right">{badge}
    <span style="color:{MUT};font-size:11px;margin-left:8px">{total} results</span>
  </span>
</div>
<table style="width:100%;border-collapse:collapse;background:{SURF}">
  <tr style="background:{SURF2}">
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left;width:50px">Rank</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left;width:200px">Prompt</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left">Preview</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:right;width:80px">Score</th>
  </tr>
  {rows_html}
</table>"""

    return header, badge, total, target_rank


# ── main render ───────────────────────────────────────────────────────────────
html = (f'<div style="background:{BG};padding:20px;border-radius:14px;'
        f'font-family:system-ui,sans-serif;max-width:1600px">'
        f'<h2 style="color:{TXT};margin:0 0 4px">ArrowSpace Spot-Check — Base vs MMR+Salience</h2>'
        f'<p style="color:{MUT};font-size:12px;margin:0 0 18px">'
        f'α={ALPHA} · top-3 and bottom-3 of {N_RETRIEVE} · '
        f'<span style="color:{BLUE}">■ Base</span>  '
        f'<span style="color:{AMBER}">■ MMR+Salience (λ={LAM}, sal={SAL_WEIGHT})</span>'
        f'  <span style="color:{TEAL}">✦ = salience score</span></p>')

wins_mmr = wins_base = ties = 0

for qid in SPOT_QUERIES:
    meta_q      = idx[qid]
    row_i       = meta_q["row_index"]
    expected_pk = meta_q["expected_prompt_id"]
    query_type  = meta_q.get("query_type", "")
    query_text  = meta_q.get("query_text", meta_q.get("query", ""))
    q_vec       = q_embs[row_i].astype(np.float64)

    # ── FIX: single search call — reuse hits_base as candidates for MMR ──────
    hits_base  = aspace.search(q_vec, gl, ALPHA)
    candidates = hits_base   # same pool, different reranking strategy
    hits_mmr   = mmr_rerank(candidates, embs, ids, lam=LAM, k=N_RETRIEVE,
                             sal_weight=SAL_WEIGHT, sal_map=salience_map)

    tbl_base, badge_base, total_base, rank_base = _build_table(
        hits_base, ids, expected_pk, N_RETRIEVE,
        label="Base — semantic only", color=BLUE)
    tbl_mmr, badge_mmr, total_mmr, rank_mmr = _build_table(
        hits_mmr, ids, expected_pk, N_RETRIEVE,
        label="MMR+Salience reranked", color=AMBER, sal_map=salience_map)

    # ── win indicator ─────────────────────────────────────────────────────────
    r_b = rank_base if rank_base else 999
    r_m = rank_mmr  if rank_mmr  else 999
    if r_m < r_b:
        win_label = f'<span style="color:{GRN};font-weight:700">▲ MMR wins (#{r_m} vs #{r_b})</span>'
        wins_mmr += 1
    elif r_b < r_m:
        win_label = f'<span style="color:{BLUE};font-weight:700">▲ BASE wins (#{r_b} vs #{r_m})</span>'
        wins_base += 1
    else:
        win_label = f'<span style="color:{MUT}">= TIE (both #{r_b})</span>'
        ties += 1

    query_block = ""
    if query_text:
        query_block = (f'<div style="background:{BG};border-left:3px solid {TEAL};'
                       f'padding:8px 12px;margin:8px 0 0;font-size:12px;color:{TXT};'
                       f'font-style:italic">'
                       f'<span style="color:{TEAL};font-style:normal;font-weight:600;'
                       f'font-size:10px;display:block;margin-bottom:3px">QUERY</span>'
                       f'{query_text}</div>')

    html += f"""
<div style="border:1px solid {BDR};border-radius:10px;overflow:hidden;margin-bottom:20px">
  <div style="background:{SURF2};padding:10px 14px;display:flex;
      align-items:center;gap:10px;flex-wrap:wrap">
    <span style="font-size:15px;font-weight:700;color:{TXT}">{qid}</span>
    <span style="background:{BG};color:{MUT};font-size:11px;padding:2px 8px;
        border-radius:999px">{query_type}</span>
    <span style="font-size:11px;color:{MUT}">target:
      <code style="color:{ORG};background:{BG};padding:1px 6px;
          border-radius:3px">{expected_pk}</code>
    </span>
    <span style="margin-left:auto;font-size:12px">{win_label}</span>
  </div>
  {query_block}
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:0;
      border-top:1px solid {BDR}">
    <div style="border-right:1px solid {BDR}">{tbl_base}</div>
    <div>{tbl_mmr}</div>
  </div>
</div>"""

# ── summary scoreboard ────────────────────────────────────────────────────────
total_q = len(SPOT_QUERIES)
html += f"""
<div style="background:{SURF2};border-radius:10px;padding:16px 20px;
    display:flex;gap:32px;align-items:center;flex-wrap:wrap">
  <span style="color:{TXT};font-weight:700;font-size:14px">Scoreboard</span>
  <span style="color:{AMBER};font-weight:700">MMR+Sal wins: {wins_mmr}/{total_q}</span>
  <span style="color:{BLUE};font-weight:700">Base wins: {wins_base}/{total_q}</span>
  <span style="color:{MUT}">Ties: {ties}/{total_q}</span>
</div>"""

html += '</div>'
display(HTML(html))

### **5.5 Retrieval Interpretation**

| Metric | Result |
|---|---|
| Base wins | 3 / 6 |
| MMR+Sal wins | 0 / 6 |
| Ties | 3 / 6 |
| Target dropped out of top-10 | 1 / 6 |

**1. Base retrieval is the stronger baseline**
By using **pure cosine similarity** over the **ArrowSpace graph-Laplacian** embedding wins/ties in all the 6 queries. This shows that the base embedding has strong meaning, making it hard to beat with a fine-tuned retriever in a small test. 

**2. Salience signal introduces popularity bias**
In `q_02`, the target (`pk_19098`) is ranked **#3** by base but drops out of the top-10 after **MMR+Salience reranking**. The reranker boots the high-upvote items (`pk_14475`, 2,897 upvotes, 0.67) that are semantically adjacent but not intent-matched. This is a common popularity bias, where the salience score conflates engagement volume with relevance.

**3. MMR diversifies correctly but over-penalises the target.**
In `q_01`, MMR moves the target from **#2 to #4** by reducing redundancy, but wrongly sees `pk_18717`(a recruiting-side salary script) as too similar to chosen sales negotiation prompts. The margin between score distributions is **thin (~0.01)**, suggesting **λ=1.0** is too aggressive. A softer **λ ∈ [0.5, 0.7]** would preserve more semantic score weight and avoid this over-penalisation.

**4. Both methods handle unambiguous queries equally.**
`q_03` (salary disclosure law, Florida) is a tie at **#1** for **both pipelines**. Relevance and MMR penalties have no effect when the query is very specific and the correct document is semantically distant from all others in the corpus. Since  the base score prevails and both pipelines converge, this confirms the quality of the embedding for niche and industry-specific queries.